In [1]:
import pandas as pd
import json
from tqdm import tqdm

from utils.subgraph_functions import  prune_triples, restore_full_triples, get_full_uri
from utils.helper_functions import read_jsonl_file, save_as_jsonl, decode_underscored_unicode


In [2]:
split = 'test'
directory = f"/home/ec2-user/preetam_experiments/"

df = pd.read_json(f"{directory}/outputs/{split}/cleaned_subgraph_df_{split}.json", orient="records")

res_file = read_jsonl_file(f"{directory}/batch_processing_files/{split}/results/llm_requests_{split}_out.jsonl")

In [ ]:
df

,subgraph_Steiner,subgraph_Steiner_length,subgraph_Steiner_largest_connected,subgraph_Steiner_largest_connected_length,QID,unwanted_filter_flag,unwanted_percentage
0,[[http://yago-knowledge.org/resource/Old__u002...,30,[[http://yago-knowledge.org/resource/Camelidae...,15,97855,False,0.000000
1,[[http://yago-knowledge.org/resource/Barry_Ban...,13,[[http://yago-knowledge.org/resource/Barry_Ban...,11,39548,False,0.000000
2,[[http://yago-knowledge.org/resource/Joseph_Ma...,16,[[http://yago-knowledge.org/resource/Joseph_Ma...,14,118880,False,0.000000
3,[[http://yago-knowledge.org/resource/Joe_Keena...,17,[[http://yago-knowledge.org/resource/Joe_Keena...,13,31930,False,0.000000
4,[[http://yago-knowledge.org/resource/Standards...,20,[[http://yago-knowledge.org/resource/Mzla_Tech...,17,97616,False,0.000000
...,...,...,...,...,...,...,...
19995,[[http://yago-knowledge.org/resource/Building_...,44,[[http://yago-knowledge.org/resource/Building_...,33,86508,True,45.454545
19996,[[http://yago-knowledge.org/resource/Reggie_Mi...,33,[[http://yago-knowledge.org/resource/Reggie_Mi...,33,50516,True,45.454545
19997,[[http://yago-knowledge.org/resource/Emma_Thom...,42,[[http://yago-knowledge.org/resource/Emma_Thom...,33,120462,True,45.454545
19998,"[[http://yago-knowledge.org/resource/China, ht...",17,"[[http://yago-knowledge.org/resource/China, ht...",11,94732,True,45.454545


In [4]:
data_dict = {
    'id':[],
    'question_num':[],
    'question':[],
    'answer':[],
    'answer_readable':[],
    'answer_uri':[],
    'supporting_path':[],
    'supporting_path_uri':[],
    'subgraph':[],
    'subgraph_size':[]}

In [6]:
failed_idx = []
for idx in tqdm(range(len(df))): 
    try:
        response = json.loads(res_file[idx]['modelOutput']['content'][0]['text'])
        if response['valid_qa_pairs']:
            qa_pairs = response['qa_pairs']
            for qnum in range(len(qa_pairs)):
                supporting_path_uri = restore_full_triples(qa_pairs[qnum]['supporting_path'], df.iloc[idx]['subgraph_Steiner_largest_connected'])
                if supporting_path_uri[1] == False:
                    failed_idx.append(f'QID: {df.iloc[idx]["QID"]}, QNUM: {qnum}')
                    # print(f'QID: {df.iloc[idx]["QID"]}, QNUM: {qnum}')
                    # print(f'Failed supporting path: {qa_pairs[qnum]["supporting_path"]}')
                    continue
                
                answer_node = get_full_uri(qa_pairs[qnum]['answer'], supporting_path_uri[0])
                if answer_node == None:
                    failed_idx.append(f'QID: {df.iloc[idx]["QID"]}, QNUM: {qnum}')
                    # print(f'QID: {df.iloc[idx]["QID"]}, QNUM: {qnum}')
                    # print(f'Failed answer node: {qa_pairs[qnum]["answer"]}')
                    continue
                
                
                data_dict['answer'].append(qa_pairs[qnum]['answer'])
                data_dict['answer_uri'].append(answer_node)
                data_dict['answer_readable'].append(decode_underscored_unicode(qa_pairs[qnum]['answer']))
                
                data_dict['supporting_path'].append(qa_pairs[qnum]['supporting_path'])
                data_dict['supporting_path_uri'].append(supporting_path_uri[0])
                    
                data_dict['id'].append(df.iloc[idx]['QID'])
                data_dict['question_num'].append(qnum)
                data_dict['question'].append(qa_pairs[qnum]['question'])

                
                data_dict['subgraph'].append(df.iloc[idx]['subgraph_Steiner_largest_connected'])
                data_dict['subgraph_size'].append(df.iloc[idx]['subgraph_Steiner_largest_connected_length'])
    except:
        failed_idx.append(f'QID: {df.iloc[idx]["QID"]}')
        # print(f'QID: {df.iloc[idx]["QID"]}')
        continue
    

100%|██████████| 20000/20000 [00:13<00:00, 1529.21it/s]


In [7]:
len(failed_idx)

3461

In [9]:
df_n = pd.DataFrame(data_dict)

In [10]:
df_n#[40:60]

,id,question_num,question,answer,answer_readable,answer_uri,supporting_path,supporting_path_uri,subgraph,subgraph_size
0,97855,0,"What family does the alpaca belong to, conside...",Camelidae,Camelidae,http://yago-knowledge.org/resource/Camelidae,"[{'subject': 'Alpaca', 'predicate': 'parentTax...","[[http://yago-knowledge.org/resource/Alpaca, h...",[[http://yago-knowledge.org/resource/Camelidae...,15
1,97855,1,Which broader taxonomic group includes both th...,Camelidae,Camelidae,http://yago-knowledge.org/resource/Camelidae,"[{'subject': 'Guanaco', 'predicate': 'parentTa...","[[http://yago-knowledge.org/resource/Guanaco, ...",[[http://yago-knowledge.org/resource/Camelidae...,15
2,97855,2,What is the highest-level taxonomic group that...,Even-toed_ungulate,Even-toed ungulate,http://yago-knowledge.org/resource/Even-toed_u...,"[{'subject': 'Chevrotain', 'predicate': 'paren...",[[http://yago-knowledge.org/resource/Chevrotai...,[[http://yago-knowledge.org/resource/Camelidae...,15
3,39548,0,Which football club owned Villa Park and parti...,Aston_Villa_F_u002E_C_u002E_,Aston Villa F.C.,http://yago-knowledge.org/resource/Aston_Villa...,"[{'subject': 'Villa_Park', 'predicate': 'owned...",[[http://yago-knowledge.org/resource/Villa_Par...,[[http://yago-knowledge.org/resource/Barry_Ban...,11
4,39548,1,Which player was a member of both Aston Villa ...,Gary_Cahill,Gary Cahill,http://yago-knowledge.org/resource/Gary_Cahill,"[{'subject': 'Gary_Cahill', 'predicate': 'memb...",[[http://yago-knowledge.org/resource/Gary_Cahi...,[[http://yago-knowledge.org/resource/Barry_Ban...,11
...,...,...,...,...,...,...,...,...,...,...
56977,120462,2,Which male actor appeared in House and shares ...,Nathan_Kress,Nathan Kress,http://yago-knowledge.org/resource/Nathan_Kress,"[{'subject': 'House__u0028_TV_series_u0029_', ...",[[http://yago-knowledge.org/resource/House__u0...,[[http://yago-knowledge.org/resource/Emma_Thom...,33
56978,94732,0,Which movie director shares the same nationali...,Sammo Hung,Sammo Hung,http://yago-knowledge.org/resource/Sammo_Hung,"[{'subject': 'South_Korea', 'predicate': 'neig...",[[http://yago-knowledge.org/resource/South_Kor...,"[[http://yago-knowledge.org/resource/China, ht...",11
56979,94732,1,In which historical kingdom was the mother of ...,Joseon,Joseon,http://yago-knowledge.org/resource/Joseon,[{'subject': 'Royal_Noble_Consort_Yeongbin_Yi'...,[[http://yago-knowledge.org/resource/Royal_Nob...,"[[http://yago-knowledge.org/resource/China, ht...",11
56980,65177,0,Which language would Frank Chodorov's organiza...,English_language_generic_instance,English language generic instance,http://yago-knowledge.org/resource/English_lan...,"[{'subject': 'Frank_Chodorov', 'predicate': 'n...",[[http://yago-knowledge.org/resource/Frank_Cho...,[[http://yago-knowledge.org/resource/Prince__u...,11
